# JSONL LLM Trace Explorer

Interactive viewer for task-decomposition JSONL event logs. Run all cells, enter the JSONL file path when prompted, and the trace displays below. Supports any similarly structured JSONL file. No network access required.

**Install (only if needed):** `uv add --dev jupyterlab ipykernel ipywidgets pandas`  
For VS Code, open this notebook with a Python/Jupyter kernel. If widgets do not render, update the Jupyter and ipywidgets extensions.

In [1]:
from pathlib import Path
import json, html
import pandas as pd
from IPython.display import display, HTML, JSON, Markdown, clear_output
import ipywidgets as widgets

TRACE_PATH = Path(input('Enter the JSONL trace file path: ').strip()).expanduser()
if not TRACE_PATH.is_file():
    raise FileNotFoundError(f'Trace file not found: {TRACE_PATH}')


In [2]:
def load_trace(path):
    events, issues = [], []
    with Path(path).open(encoding='utf-8-sig') as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip(): continue
            try:
                item = json.loads(line)
                if not isinstance(item, dict):
                    issues.append(f'Line {line_number}: expected a JSON object')
                    continue
                item['_line'] = line_number
                events.append(item)
            except json.JSONDecodeError as e:
                issues.append(f'Line {line_number}: {e}')
    return events, issues

def stage_of(e): return e.get('stage') or '(session)'
def status_of(e):
    if e.get('result') is not None: return str(e['result'])
    if e.get('event') == 'execution.summary': return str(e.get('data',{}).get('status','—'))
    return '—'
def as_table(events):
    return pd.DataFrame([{
        'line':e['_line'], 'timestamp':e.get('timestamp',''),
        'stage':stage_of(e), 'event':e.get('event',''),
        'status':status_of(e),
        'latency_ms':e.get('data',{}).get('latency_ms'),
        'model':e.get('data',{}).get('model') or e.get('data',{}).get('response_metadata',{}).get('model','') if isinstance(e.get('data',{}).get('response_metadata',{}),dict) else '',
    } for e in events])

events, issues = load_trace(TRACE_PATH)
df = as_table(events)
print(f'Loaded {len(events)} events from {TRACE_PATH.name}; parse issues: {len(issues)}')
if issues: display(pd.DataFrame({'issue':issues}))


Loaded 21 events from task_decomposition_2026-10-14_234344.jsonl; parse issues: 0


## Execution summary

In [3]:
display(pd.DataFrame([{'events':len(events), 'stages':df.loc[df.stage!='(session)','stage'].nunique(),
                       'failures':int(df.status.isin(['failure','error','failed']).sum()),
                       'total_llm_latency_s':round(pd.to_numeric(df.latency_ms,errors='coerce').sum()/1000,2)}]))
display(df.groupby('stage',sort=False).agg(events=('event','size'), last_status=('status','last'), llm_latency_ms=('latency_ms','sum')).reset_index())

def usage_rows(events):
    rows=[]
    for e in events:
        usage=e.get('data',{}).get('usage')
        if e.get('event')!='llm.response' or not isinstance(usage,dict):continue
        rows.append({'stage':stage_of(e),**{k:v for k,v in usage.items() if isinstance(v,(int,float))}})
    return pd.DataFrame(rows)
usage=usage_rows(events)
if not usage.empty:
    display(Markdown('### Token usage by response'))
    display(usage)


,events,stages,failures,total_llm_latency_s
0,21,4,0,16.6


,stage,events,last_status,llm_latency_ms
0,(session),1,—,0.000
1,operational_decomposition,5,success,8607.652
2,semantic_evaluation,4,accept,1399.793
3,retain_remove_classification,5,success,4781.661
4,added_work_classification,6,success,1815.468


### Token usage by response

,stage,input_tokens,output_tokens,total_tokens
0,operational_decomposition,786,674,1460
1,semantic_evaluation,2058,37,2095
2,retain_remove_classification,1384,521,1905
3,added_work_classification,2140,72,2212


## Interactive trace browser
Select an event to inspect its full payload. Long prompts and model outputs are displayed in collapsible panels. You can also inspect any nested object as formatted JSON.

In [ ]:
def pretty(value):
    if isinstance(value,str):
        try: value=json.loads(value)
        except (ValueError,TypeError): return value
    return json.dumps(value,ensure_ascii=False,indent=2,default=str)

def panel(title,value,open_by_default=False):
    content=html.escape(pretty(value))
    return f'<details {"open" if open_by_default else ""}><summary style="cursor:pointer;font-weight:600;padding:7px">{html.escape(title)}</summary><pre style="white-space:pre-wrap;overflow-wrap:anywhere;background:#f5f7fa;color:#182230;padding:12px;border-radius:6px;max-height:520px;overflow:auto">{content}</pre></details>'

def show_event(e):
    data=e.get('data') or {}
    title=f"Line {e['_line']} · {e.get('event','')} · {stage_of(e)} · {status_of(e)}"
    display(Markdown('### '+title))
    sections=[]
    if isinstance(data,dict):
        if 'messages' in data:
            for idx,msg in enumerate(data['messages']):
                sections.append(panel(f"Message {idx+1} — {msg.get('role','')}",msg.get('content','')))
        for key in ('parsed_response','response','raw_output','errors','error','usage','parameters','structured_output_schema','raw_response'):
            if key in data: sections.append(panel(key,data[key],key in ('errors','error','parsed_response')))
        remaining={k:v for k,v in data.items() if k not in ('messages','parsed_response','response','raw_output','errors','error','usage','parameters','structured_output_schema','raw_response')}
        if remaining: sections.append(panel('Other data',remaining))
    sections.append(panel('Complete event JSON',e))
    display(HTML(''.join(sections)))

def trace_browser(events):
    stages=['All']+list(dict.fromkeys(stage_of(e) for e in events))
    types=['All']+sorted(set(e.get('event','') for e in events))
    stage=widgets.Dropdown(options=stages,description='Stage:',layout=widgets.Layout(width='390px'))
    kind=widgets.Dropdown(options=types,description='Event:',layout=widgets.Layout(width='390px'))
    status=widgets.Dropdown(options=['All']+sorted(set(status_of(e) for e in events)),description='Status:')
    selector=widgets.Select(options=[],description='Entries:',rows=11,layout=widgets.Layout(width='100%',height='235px'))
    summary=widgets.Output(); details=widgets.Output()
    def update_list(*_):
        matches=[e for e in events if (stage.value=='All' or stage_of(e)==stage.value) and (kind.value=='All' or e.get('event')==kind.value) and (status.value=='All' or status_of(e)==status.value)]
        selector.options=[(f"{e['_line']:>3} | {e.get('event',''):<28} | {stage_of(e):<30} | {status_of(e)}",e['_line']) for e in matches]
        with summary:
            clear_output(wait=True)
            display(Markdown(f'**{len(matches)} matching events**'))
    def update_detail(change):
        with details:
            clear_output(wait=True)
            match=next((e for e in events if e['_line']==selector.value),None)
            if match: show_event(match)
    for w in (stage,kind,status): w.observe(update_list,names='value')
    selector.observe(update_detail,names='value')
    display(widgets.HBox([stage,kind,status]),summary,selector,details)
    update_list()

trace_browser(events)


Output()

Select(description='Entries:', layout=Layout(height='235px', width='100%'), options=(), rows=11, value=None)

Output()

## Structured output: operational subtasks
Extracts the decomposition from parsed events, validation results, or response text, preferring parsed output.

In [ ]:
def find_subtasks(obj):
    if isinstance(obj,dict):
        subtasks=obj.get('operational_subtasks')
        if isinstance(subtasks,list):return obj
        for v in obj.values():
            result=find_subtasks(v)
            if result is not None:return result
    elif isinstance(obj,list):
        for v in obj:
            result=find_subtasks(v)
            if result is not None:return result
    elif isinstance(obj,str):
        try:return find_subtasks(json.loads(obj))
        except (ValueError,TypeError):pass
    return None

candidates=sorted(events,key=lambda e: 0 if e.get('event')=='llm.parsed' else 1)
decomposition=next((found for e in candidates if (found:=find_subtasks(e.get('data',{}))) is not None),None)
if decomposition:
    task=decomposition.get('task',{})
    display(Markdown('**Task:** '+str(task.get('task_name','Unknown'))))
    subtasks=pd.DataFrame(decomposition['operational_subtasks'])
    if 'depends_on' in subtasks: subtasks['depends_on']=subtasks['depends_on'].apply(lambda x:', '.join(map(str,x)) if isinstance(x,list) else str(x))
    display(subtasks.drop(columns=['provenance_refs'],errors='ignore').style.set_properties(**{'text-align':'left','white-space':'pre-wrap'}))
else: print('No operational_subtasks object found in this trace.')


## Validation diagnostics

In [ ]:
diagnostics=[]
for e in events:
    if e.get('event','').startswith('validation.') or e.get('event')=='execution.summary':
        d=e.get('data') or {}
        diagnostics.append({'line':e['_line'],'stage':stage_of(e),'event':e.get('event'),'result':status_of(e),
                            'errors':pretty(d.get('errors',d.get('error','')))[:2500]})
if diagnostics: display(pd.DataFrame(diagnostics).style.set_properties(**{'white-space':'pre-wrap','text-align':'left'}))
else: print('No validation events found.')
